# Group Stats Visualization

In [ ]:
from datetime import datetime
from pathlib import Path
import re

import jieba
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from wordcloud import STOPWORDS, WordCloud

from monthly_report.analysis import (
    build_message_dataframe,
    grouped_message_counts,
    monthly_platform_counts,
    top_n_with_other,
)
from monthly_report.config import ReportConfig
from monthly_report.loaders import load_all_messages

In [ ]:
from matplotlib import font_manager as fm
font_path = r"<path to preferred font>"
font = fm.FontProperties(fname = font_path)
# title_font = fm.FontProperties(fname = font_path, size=42)
plt.rcParams['font.family'] = font.get_name()

In [ ]:
BASE_DIR = Path(r"<path to base directory>")
NAMEMAP_FILE = BASE_DIR / "<path to namemap>.json"
RECENT_CUTOFF = datetime.fromisoformat("2025-11-01")
OLD_CUTOFF = datetime.fromisoformat("2025-02-01")
TOP_N = 16

ERASED_IDS = {
    "q(1234567890)", # QQ
    "user76543210123", # TG
}

config = ReportConfig(base_dir=BASE_DIR, namemap_file=NAMEMAP_FILE, erased_ids=ERASED_IDS)

messages = load_all_messages(config)
df = build_message_dataframe(messages)

In [ ]:
counts_all = grouped_message_counts(df)
counts_recent = grouped_message_counts(df, since=RECENT_CUTOFF)
pie_all = top_n_with_other(counts_all, TOP_N)
pie_recent = top_n_with_other(counts_recent, TOP_N)

print(f"messages={len(messages):,}, members={df['name'].nunique():,}")

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(16, 7), layout='constrained')
pie_all.plot.pie(autopct='%1.1f%%', ax=axes[0], label='')
axes[0].set_title('Message Distribution: All')
pie_recent.plot.pie(autopct='%1.1f%%', ax=axes[1], label='')
axes[1].set_title(f'Message Distribution: Since {RECENT_CUTOFF.date()}')
plt.show()

In [ ]:
today = pd.Timestamp.today().normalize()
include_current_month = today.day > (today.days_in_month / 2)
end_period = today.to_period("M") if include_current_month else (today.to_period("M") - 1)
month_periods = pd.period_range(end=end_period, periods=9, freq="M")

month_bucket = df["date"].dt.to_period("M")
fig, axes = plt.subplots(3, 3, figsize=(15, 15), layout="constrained")

for ax, period in zip(axes.flat, month_periods):
    month_frame = df[month_bucket == period]
    counts = grouped_message_counts(month_frame)

    if counts.empty:
        ax.text(0.5, 0.5, "No messages", ha="center", va="center")
        ax.set_axis_off()
    else:
        pie = top_n_with_other(counts, n_top=TOP_N)
        pie.plot.pie(
            autopct="%1.1f%%",
            ax=ax,
            label="",
            textprops={"fontsize": 8},
            startangle=90,
        )
        ax.set_ylabel("")

    ax.set_title(period.strftime("%Y-%m"))

window_start = month_periods[0].strftime("%Y-%m")
window_end = month_periods[-1].strftime("%Y-%m")
fig.suptitle(
    f"Sender Distribution by Month ({window_start} to {window_end})"
    f"\nInclude current month: {include_current_month}",
    fontsize=14,
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
monthly = df.groupby(pd.Grouper(key='date', freq='ME'))['msg_len'].sum()
monthly.index = monthly.index.strftime('%Y-%m')
monthly.plot(kind='bar', ax=ax)
ax.set_title('# Message by Month')
ax.set_xlabel('Month')
plt.show()

In [ ]:
platform_monthly = monthly_platform_counts(df, since=OLD_CUTOFF)
ax = platform_monthly.plot(kind='bar', stacked=True, figsize=(10, 5))
ax.set_title('Platform Usage by Month')
ax.set_xlabel('Month')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))

In [ ]:
sns.histplot(df['date'].dt.hour, bins=np.arange(0, 25))
plt.title('Hour Distribution')
plt.xlabel('Hour')

In [ ]:
# Word-cloud text preprocessing (ported from legacy notebook)
extra_sw = ["图片", "表情", "一个", "好像", "感觉", "说"]
STOPWORDS.update(extra_sw)

stopword_candidates = [
    Path("baidu_stopwords.txt"),
]
for sw_path in stopword_candidates:
    if sw_path.exists():
        with sw_path.open("r", encoding="utf-8") as f:
            STOPWORDS.update(line.strip() for line in f if line.strip())
        break

userdict_candidates = [
    Path("custom_dict.txt"),
]
for dict_path in userdict_candidates:
    if dict_path.exists():
        jieba.load_userdict(str(dict_path))
        break

def split_words(messages: pd.Series) -> str:
    s = "\n".join(messages.astype(str).tolist())
    s = " ".join(jieba.lcut(s)).strip()
    return s if s else "Error"

In [ ]:
space_str = split_words(df[df["date"] > RECENT_CUTOFF]["message"])
wc = WordCloud(
    width=1800,
    height=1800,
    background_color="black",
    max_words=200,
    stopwords=STOPWORDS,
    collocations=False,
    font_path=font_path,
    max_font_size=300,
    relative_scaling=0.4,
    random_state=50,
    scale=2,
).generate(space_str)
plt.figure(figsize=(12, 8))
plt.imshow(wc)
plt.title(f"Word Cloud: Messages Since {RECENT_CUTOFF.date()}")
plt.axis("off")

In [ ]:
month_bucket = df["date"].dt.to_period("M")
fig, axes = plt.subplots(3, 3, figsize=(15, 15), layout="constrained")

for ax, period in zip(axes.flat, month_periods):
    month_messages = df.loc[month_bucket == period, "message"]
    month_text = split_words(month_messages)

    if month_text.strip() and month_text != "Error":
        month_wc = WordCloud(
            width=1200,
            height=1200,
            background_color="black",
            max_words=133,
            stopwords=STOPWORDS,
            collocations=False,
            font_path=font_path,
            max_font_size=270,
            relative_scaling=0.4,
            random_state=50,
            scale=2,
        ).generate(month_text)
        ax.imshow(month_wc)
    else:
        ax.text(0.5, 0.5, "No messages", ha="center", va="center")

    ax.set_axis_off()
    ax.set_title(period.strftime("%Y-%m"), fontsize=10)

window_start = month_periods[0].strftime("%Y-%m")
window_end = month_periods[-1].strftime("%Y-%m")
fig.suptitle(
    f"Monthly Word Clouds ({window_start} to {window_end})"
    f"\nInclude current month: {include_current_month}",
    fontsize=14,
    y=1.02,
 )

plt.show()

In [ ]:
top_chatters = grouped_message_counts(df, since=RECENT_CUTOFF).head(TOP_N).index.tolist()
fig, axes = plt.subplots(4, 4, figsize=(15, 15), layout="constrained")

for ax, name in zip(axes.flat, top_chatters):
    chatter_messages = df.loc[df["name"] == name, "message"]
    chatter_text = split_words(chatter_messages)

    if chatter_text.strip() and chatter_text != "Error":
        chatter_wc = WordCloud(
            width=900,
            height=900,
            background_color="black",
            max_words=133,
            stopwords=STOPWORDS,
            collocations=False,
            font_path=font_path,
            max_font_size=200,
            relative_scaling=0.4,
            random_state=50,
            scale=2,
        ).generate(chatter_text)
        ax.imshow(chatter_wc)
    else:
        ax.text(0.5, 0.5, "No messages", ha="center", va="center")

    chatter_count = int(((df["name"] == name) & (df["date"] >= RECENT_CUTOFF)).sum())
    ax.set_title(f"{name} ({chatter_count:,})", fontsize=9)
    ax.set_axis_off()

for ax in axes.flat[len(top_chatters):]:
    ax.set_axis_off()

fig.suptitle("Word Clouds: Top 16 Chatters", fontsize=16, y=1.01)
plt.show()